---
# Chapter 10 — What Matters Right Now?

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 10: What Matters Right Now? |
| Central question | How does the present work frame change which memories are selected? |
| Main concepts | ProjectFrame, WorkFrame, ContextBundle, ContextTrace, Frame-conditioned relevance |
| Implementation | context_frames |
| Experiment | ch10-20260920T163314Z-context-frames |
| Evidence status | Book result: core as explicit control, conditional when inferred |
| Depends on | Chapter 3 (baseline), Chapter 2 (instrument) |

---

## What this notebook demonstrates

This chapter introduces **explicit framing** as a control and context-selection mechanism. The notebook:

1. **Loads the frozen context frames run** (`ch10-20260920T163314Z-context-frames`)
2. **Shows ProjectFrame and WorkFrame** changing memory selection
3. **Inspects ContextBundle and ContextTrace** — the audit trail
4. **Demonstrates the central results**: same query under different goals produces different memory; declared WorkFrames materially changed context selection; project framing eliminated cross-project leakage
5. **Shows the hazard**: automatically inferred frames nearly erased the gain; a wrong frame could be worse than no frame

> **Evidence status**: Book result. Frame-conditioned selection is core as explicit control, conditional when inferred.

## The chapter question

> **What from the past matters right now?**

This question changed the book. Chapter 10 introduced ProjectFrame, WorkFrame, ContextBundle and ContextTrace and measured them directly against strong query-only RAG.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(10)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 10 Concepts")

## Load the frozen context frames run

In [ ]:
from notebooks.memory._support import load_frozen_run

run = load_frozen_run("ch10-20260920T163314Z-context-frames")
metrics = run["metrics"]

print(f"Run ID: {run['run_id']}")
render_table(
    [{"Condition": cond,
      "MUST recall": m["must_include_recall"],
      "Harmful admitted": m["harmful_admission"],
      "Cross-project leakage": m["cross_project_leakage"],
      "Bundle tokens": m["bundle_tokens"],
      "Label": m["label"]}
     for cond, m in metrics["ladder_mean"].items()],
    "Frozen ch10 ladder means")
print("\nCross-project leakage by condition "
      "(query-only leaks, framed conditions do not):")
print(metrics["cross_project_leakage"])

## ProjectFrame and WorkFrame

The frame representation from `context_frames`:

In [ ]:
from context_frames import ProjectFrame, WorkFrame, ContextBundle, ContextTrace

print("context_frames loaded")
print(f"ProjectFrame fields: {list(ProjectFrame.__dataclass_fields__.keys())}")
print(f"WorkFrame fields: {list(WorkFrame.__dataclass_fields__.keys())}")
print(f"ContextBundle fields: {list(ContextBundle.__dataclass_fields__.keys())}")
print(f"ContextTrace fields: {list(ContextTrace.__dataclass_fields__.keys())}")

## Create frames and see how they change selection

In [ ]:
from context_frames import fixtures as FX

project = FX.PF_MEMORY_BOOK()
print(f"ProjectFrame: {project.frame_id()} — {project.purpose}")
print(f"Known work types: {project.known_work_types()}")

t_pub = FX.T1_PUBLICATION()
t_arch = FX.T1_ARCHITECTURE()
print(f"\nShared query: {t_pub.query}")
for t in (t_pub, t_arch):
    wf = t.work_frame
    print(f"\n{t.task_id} [{wf.work_type}] (derivation={wf.derivation}):")
    print(f"  objective: {wf.objective}")
    print(f"  signals: {[(s.signal_id, s.kind) for s in wf.signals]}")
    print(f"  MUST ledger: {[u for u, g in t.ledger.items() if g == 'MUST'][:4]}")

## Frame-conditioned selection (conceptual demonstration)

The same query under different WorkFrames should produce different memory selections.

In [ ]:
# Same query, two goals, two different memories. The build runs fully
# offline (BM25-only retriever, no embedding service, no model calls).
from context_frames import build_context
from context_frames import corpus as CORPUS
from context_frames.retrieval import HybridRetriever

units = CORPUS.by_id()
retriever = HybridRetriever(CORPUS.corpus())
print("Retriever:", retriever.describe())

bundles = {}
for name, task in (("publication", t_pub), ("architecture", t_arch)):
    bundle, trace = build_context("C4", task.query, project,
                                  task.work_frame, retriever, units,
                                  budget_tokens=1500)
    bundles[name] = (bundle, trace)
    print(f"\n=== {name} [{task.work_frame.work_type}] ===")
    print(f"admitted {len(bundle.items)} items, {bundle.tokens} tokens, "
          f"digest {bundle.digest()}")
    for item in bundle.items[:8]:
        print(f"  {item.item_id} [{item.kind}]")
    print(f"trace: {len(trace.admitted())} admitted / "
          f"{len(trace.rejected())} rejected")

pub_ids = {i.item_id for i in bundles["publication"][0].items}
arch_ids = {i.item_id for i in bundles["architecture"][0].items}
print(f"\nShared items: {len(pub_ids & arch_ids)} of "
      f"{len(pub_ids | arch_ids)} total")

## The central results (from chapter)

- **Same query under different goals → different memory** (implementation vs historical-review)
- **Declared WorkFrames materially changed context selection**
- **Project framing eliminated cross-project leakage** in controlled suite
- **Temporal/evidence/open-loop signals removed harmful stale material**
- **Answer correctness did NOT improve over strong RAG baseline**
- **Automatically inferred frames nearly erased the gain**
- **A wrong frame could be worse than no frame**
- **A frame that only reranks cannot recover evidence that frame-blind retrieval never proposed**

## ContextTrace — the audit trail

Every selection/rejection is recorded for auditability.

In [ ]:
# The trace explains every rejection, not just the admissions.
bundle, trace = bundles["publication"]
rejected = trace.rejected()
print(f"Example rejection: {rejected[0].unit_id}")
print("  decision:", rejected[0].decision)
print("  reason codes:", rejected[0].reason_codes)
print("  trace.why():", trace.why(rejected[0].unit_id)["reason_codes"])
print(f"\nStages: {[(s.get('stage'), s.get('kept', s.get('ranked', ''))) for s in trace.stages]}")
print(f"Bundle digest: {bundle.digest()} ({bundle.tokens} tokens)")

## What this establishes

- **Explicit framing is a control and context-selection mechanism** (core)
- **Frame establishment itself is a hazard** — wrong/inferred frames can destroy recall
- **Answer correctness didn't improve over strong RAG** — frames help selection, not reasoning
- **Cross-project leakage eliminated** — ProjectFrame is effective isolation
- **Inferred frames conditional** — per-reader simplifications disagree and don't transfer

## What this does NOT establish

- Frames improve answer correctness (they don't)
- Automatic frame inference works reliably (it doesn't)
- Real-corpus frame extraction quality (untested)

## Try it yourself

Run the same publication task under `WF_T7_RELEASE` (release readiness) instead of the publication frame and compare the admitted bundles: goal-relevant evidence classes change, so the memory changes even though the query is identical.

In [ ]:
# TRY IT YOURSELF: run the same publication task under a release-
# readiness frame. Goal-relevant evidence classes change, so the
# bundle changes even though the query is identical.
wf_release = FX.WF_T7_RELEASE()
bundle_rel, trace_rel = build_context("C4", t_pub.query, project,
                                       wf_release, retriever, units,
                                       budget_tokens=1500)
rel_ids = {i.item_id for i in bundle_rel.items}
print(f"release frame [{wf_release.work_type}]: {len(rel_ids)} items")
print(f"overlap with publication bundle: {len(rel_ids & pub_ids)} items")
print("release-only head:", sorted(rel_ids - pub_ids)[:6])

## Where this leads next

Chapter 11 pushes beyond explicit loops to **consequences nobody wrote down** (derived obligations).

Chapter 13 resolves the frame hazard: **what happens when the present frame is wrong?** — reconciliation-derived establishment.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory\10-chapter.ipynb)